In [ ]:
%%sql -r dataframe_1
USE DATABASE EYPROJECT;
USE SCHEMA PUBLIC;

In [ ]:
req_text = """
pystac-client
planetary-computer
odc-stac
rasterio
xarray
rioxarray
geopandas
shapely
pyproj
"""

with open("/tmp/requirements_elevation.txt", "w") as f:
    f.write(req_text.strip())

print("Saved /tmp/requirements_elevation.txt")

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from tqdm import tqdm

print("Packages imported successfully.")

In [ ]:
train_df = pd.read_csv("water_quality_training_dataset.csv")
val_df = pd.read_csv("submission_template.csv")

train_coords = train_df[["Latitude", "Longitude"]].drop_duplicates()
val_coords = val_df[["Latitude", "Longitude"]].drop_duplicates()

all_coords = pd.concat([train_coords, val_coords]).drop_duplicates().reset_index(drop=True)

print("Unique coordinates:", len(all_coords))
display(all_coords.head())

In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import time
from tqdm import tqdm

SOIL_ENDPOINT = "https://rest.isric.org/soilgrids/v2.0/properties/query"

# Query top 30 cm and compute weighted averages
SOIL_PROPERTIES = ["clay", "sand", "silt", "soc", "bdod", "phh2o", "cec"]
SOIL_DEPTHS = ["0-5cm", "5-15cm", "15-30cm"]
SOIL_WEIGHTS = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}

# SoilGrids unit conversions
CONVERSION = {
    "clay": 10.0,     # g/kg -> %
    "sand": 10.0,     # g/kg -> %
    "silt": 10.0,     # g/kg -> %
    "soc": 10.0,      # dg/kg -> g/kg
    "bdod": 100.0,    # cg/cm3 -> kg/dm3
    "phh2o": 10.0,    # pH*10 -> pH
    "cec": 10.0       # mmol(c)/kg -> cmol(c)/kg
}

OUTPUT_COL_MAP = {
    "clay": "soil_clay",
    "sand": "soil_sand",
    "silt": "soil_silt",
    "soc": "soil_carbon",
    "bdod": "soil_bulk_density",
    "phh2o": "soil_ph",
    "cec": "soil_cec"
}

# IMPORTANT: do NOT call this variable 'session' because your notebook later uses session.sql(...)
http_session = requests.Session()
http_session.headers.update({"User-Agent": "Mozilla/5.0"})

def empty_soil_row(lat, lon):
    return {
        "Latitude": lat,
        "Longitude": lon,
        "soil_clay": np.nan,
        "soil_sand": np.nan,
        "soil_silt": np.nan,
        "soil_carbon": np.nan,
        "soil_bulk_density": np.nan,
        "soil_ph": np.nan,
        "soil_cec": np.nan
    }

def fetch_soilgrids_point(lat, lon, max_retries=5, sleep_seconds=1.0):
    params = [("lat", float(lat)), ("lon", float(lon))]
    for prop in SOIL_PROPERTIES:
        params.append(("property", prop))
    for depth in SOIL_DEPTHS:
        params.append(("depth", depth))
    params.append(("value", "mean"))

    for attempt in range(max_retries):
        try:
            response = http_session.get(SOIL_ENDPOINT, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()

            result = empty_soil_row(lat, lon)
            layers = payload.get("properties", {}).get("layers", [])

            for layer in layers:
                prop_name = layer.get("name")
                if prop_name not in OUTPUT_COL_MAP:
                    continue

                weighted_sum = 0.0
                weight_total = 0.0

                for depth_info in layer.get("depths", []):
                    top_depth = depth_info.get("range", {}).get("top_depth")
                    bottom_depth = depth_info.get("range", {}).get("bottom_depth")
                    label = f"{top_depth}-{bottom_depth}cm"

                    if label not in SOIL_WEIGHTS:
                        continue

                    raw_val = depth_info.get("values", {}).get("mean")
                    if raw_val is None:
                        continue

                    converted_val = raw_val / CONVERSION[prop_name]
                    w = SOIL_WEIGHTS[label]

                    weighted_sum += converted_val * w
                    weight_total += w

                if weight_total > 0:
                    result[OUTPUT_COL_MAP[prop_name]] = weighted_sum / weight_total

            time.sleep(sleep_seconds)
            return result

        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"Failed for lat={lat}, lon={lon}: {e}")
                return empty_soil_row(lat, lon)
            time.sleep(2 * (attempt + 1))

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Failed for lat={lat}, lon={lon}: {e}")
                return empty_soil_row(lat, lon)
            time.sleep(2 * (attempt + 1))

# Build coordinate list exactly from your notebook inputs
train_df = pd.read_csv("water_quality_training_dataset.csv")
val_df = pd.read_csv("submission_template.csv")

train_coords = train_df[["Latitude", "Longitude"]].drop_duplicates().copy()
val_coords = val_df[["Latitude", "Longitude"]].drop_duplicates().copy()
all_coords = pd.concat([train_coords, val_coords], axis=0).drop_duplicates().reset_index(drop=True)

print("Unique coordinates to query:", len(all_coords))

soil_rows = []
for _, row in tqdm(all_coords.iterrows(), total=len(all_coords)):
    soil_rows.append(fetch_soilgrids_point(row["Latitude"], row["Longitude"]))

soil_df = pd.DataFrame(soil_rows).drop_duplicates(subset=["Latitude", "Longitude"])

# Split back into train/validation feature files
soil_training = train_coords.merge(soil_df, on=["Latitude", "Longitude"], how="left")
soil_validation = val_coords.merge(soil_df, on=["Latitude", "Longitude"], how="left")

# Save locally
soil_training.to_csv("/tmp/soil_training.csv", index=False)
soil_validation.to_csv("/tmp/soil_validation.csv", index=False)

print("Soil extraction complete.")
print("soil_training shape:", soil_training.shape)
print("soil_validation shape:", soil_validation.shape)
print("\nMissingness in soil_training:")
print(soil_training.isna().mean().sort_values(ascending=False))

display(soil_training.head())
display(soil_validation.head())

In [ ]:
session.sql("""
PUT file:///tmp/soil_training.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()

session.sql("""
PUT file:///tmp/soil_validation.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()